# 02 — Generación de título y descripción a partir del draft

**Objetivo:** experimentar con el prompt y la salida estructurada para que Qwen3-4B genere
el *título* y la *descripción* de una propiedad a partir de su draft (características
estructuradas), garantizando JSON válido y **sin inventar datos**.

**Prerrequisito:** haber ejecutado `01_descarga_y_benchmark_qwen3.ipynb` (el GGUF ya debe
estar en `models_registry/llm/`).

**Limitación conocida del testbench:** el dataset sintético no tiene ubicación ni nombres de
amenidades (solo el conteo `n_amenities`), por lo que las descripciones saldrán genéricas.
Cuando el draft real incluya colonia/ciudad y amenidades con nombre, este mismo pipeline las
aprovechará sin cambios estructurales.

## 1. Carga del modelo y del testbench

In [12]:
import json
import time
from pathlib import Path

import pandas as pd
from llama_cpp import Llama

MODEL_PATH = Path("../../models_registry/llm/Qwen3-4B-Instruct-2507-Q4_K_M.gguf").resolve()
assert MODEL_PATH.exists(), "Ejecuta primero 01_descarga_y_benchmark_qwen3.ipynb"

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=8192,
    n_threads=4,
    verbose=False,
)

testbench = pd.read_csv("../testbench.csv", index_col=0)
print(f"{len(testbench)} propiedades en el testbench")
testbench.head(3)

23 propiedades en el testbench


,propertyType,areaM2,listedPrice,pricePerM2,bedrooms,bathrooms,parkingSpaces,constructionYear,antiguedad,condominium,n_amenities,total_images,is_anomaly,anomaly_type
0,Casa,367.6,4933433.44,13421.19,2,2.6,2,1986,40,0,2,9,False,NaN
1,Casa,284.3,5920899.26,20824.68,2,2.2,2,1971,55,0,10,16,False,NaN
2,Departamento,234.4,6035620.22,25745.79,2,1.9,1,2007,19,1,7,7,False,NaN


## 2. Draft → texto

Convierte una fila del testbench en un draft legible en español. Solo se incluyen los campos
que aportan al anuncio (se omiten `pricePerM2`, `total_images` y las columnas de anomalías).

Los baños vienen como decimales sintéticos (ej. `2.6`); se redondean al medio baño más cercano,
que es la convención inmobiliaria real.

In [13]:
def formatear_banos(valor: float) -> str:
    medios = round(valor * 2) / 2
    enteros = int(medios)
    if medios == enteros:
        return f"{enteros} baño{'s' if enteros != 1 else ''}"
    if enteros == 0:
        return "medio baño"
    return f"{enteros} baño{'s' if enteros != 1 else ''} y medio"


def draft_a_texto(row: pd.Series) -> str:
    lineas = [
        f"Tipo de propiedad: {row['propertyType']}",
        f"Superficie: {row['areaM2']:.0f} m²",
        f"Precio de lista: ${row['listedPrice']:,.0f} MXN",
        f"Recámaras: {int(row['bedrooms'])}",
        f"Baños: {formatear_banos(row['bathrooms'])}",
        # "Estacionamientos" y no "cajones de estacionamiento": el modelo copia el
        # vocabulario del draft, así que el término regional se fija desde aquí.
        f"Estacionamientos: {int(row['parkingSpaces'])}",
        f"Año de construcción: {int(row['constructionYear'])} ({int(row['antiguedad'])} años de antigüedad)",
        f"En condominio: {'sí' if row['condominium'] else 'no'}",
        f"Número de amenidades: {int(row['n_amenities'])}",
    ]
    return "\n".join(lineas)


print(draft_a_texto(testbench.iloc[0]))

Tipo de propiedad: Casa
Superficie: 368 m²
Precio de lista: $4,933,433 MXN
Recámaras: 2
Baños: 2 baños y medio
Estacionamientos: 2
Año de construcción: 1986 (40 años de antigüedad)
En condominio: no
Número de amenidades: 2


## 3. System prompt (v2 vigente)

La regla más importante para un modelo de 4B es la **prohibición de inventar**: sin ella,
el modelo decora con "cerca de escuelas y centros comerciales" aunque el draft no lo diga.
Iterar aquí y versionar los cambios (v1, v2, …) en la bitácora de la sección 7.

**v2:** vocabulario regional — "estacionamientos" en lugar de "cajones de estacionamiento"
(corregido también en `draft_a_texto`, porque el modelo copia el vocabulario del draft).

In [14]:
SYSTEM_PROMPT_V2 = """Eres un redactor inmobiliario profesional de México. Recibirás el draft de una \
propiedad (sus características estructuradas) y escribirás el anuncio para un portal inmobiliario.

Reglas estrictas:
1. Usa ÚNICAMENTE la información del draft. NO inventes características, ubicaciones, \
amenidades específicas, vistas ni cercanías que no estén en el draft.
2. Escribe en español de México, tono profesional y cálido, dirigido a compradores.
3. El título debe tener máximo 10 palabras y ser atractivo sin ser sensacionalista.
4. La descripción debe tener entre 20 y 70 palabras, en párrafos fluidos (sin listas).
5. No repitas el precio más de una vez. No uses mayúsculas sostenidas ni signos de admiración excesivos.
6. Si el número de amenidades es mayor a cero, menciónalo de forma genérica \
("cuenta con amenidades") sin nombrar amenidades específicas.
7. Usa el vocabulario inmobiliario de la región: di "estacionamientos" (nunca \
"cajones de estacionamiento" ni "plazas de garaje") y "recámaras" (nunca "habitaciones").

Responde exclusivamente con un JSON con las claves "titulo" y "descripcion"."""

ESQUEMA_ANUNCIO = {
    "type": "object",
    "properties": {
        "titulo": {"type": "string"},
        "descripcion": {"type": "string"},
    },
    "required": ["titulo", "descripcion"],
}

## 4. Generación con salida estructurada

`response_format` con `schema` hace que llama.cpp restrinja la generación con una gramática:
el JSON resultante es válido **por construcción**, no por buena voluntad del modelo.

In [15]:
def generar_anuncio(draft: str, temperature: float = 1.0, max_tokens: int = 512) -> dict:
    """Genera {titulo, descripcion} a partir del draft. Regresa también el tiempo de inferencia."""
    t0 = time.perf_counter()
    salida = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_V2},
            {"role": "user", "content": f"Draft de la propiedad:\n{draft}"},
        ],
        response_format={"type": "json_object", "schema": ESQUEMA_ANUNCIO},
        temperature=temperature,
        max_tokens=max_tokens,
    )
    anuncio = json.loads(salida["choices"][0]["message"]["content"])
    anuncio["tiempo_s"] = round(time.perf_counter() - t0, 1)
    return anuncio


# Prueba con una sola propiedad
demo = generar_anuncio(draft_a_texto(testbench.iloc[0]))
print(f"⏱ {demo['tiempo_s']} s\n")
print(f"TÍTULO: {demo['titulo']}\n")
print(demo["descripcion"])

⏱ 26.4 s

TÍTULO: Casa en zona tranquila con 2 recámaras

Casa de 368 m² construida en 1986, ubicada en un entorno privado sin condominio. Contiene dos recámaras y dos baños con medio. Cuenta con dos estacionamientos y dos amenidades. Ideal para quienes buscan un espacio familiar con estabilidad y cercanía a servicios básicos.


## 5. Corrida sobre el testbench completo

Genera el anuncio de las 23 propiedades y guarda `resultados_testbench.csv` para revisión
manual. Con ~10 tok/s en local esto tarda del orden de 10–20 minutos.

In [5]:
resultados = []
for idx, row in testbench.iterrows():
    draft = draft_a_texto(row)
    try:
        anuncio = generar_anuncio(draft)
        error = None
    except Exception as exc:  # JSON inválido o fallo de inferencia
        anuncio = {"titulo": None, "descripcion": None, "tiempo_s": None}
        error = str(exc)
    resultados.append(
        {
            "idx": idx,
            "propertyType": row["propertyType"],
            "areaM2": row["areaM2"],
            "titulo": anuncio["titulo"],
            "descripcion": anuncio["descripcion"],
            "n_palabras_titulo": len(anuncio["titulo"].split()) if anuncio["titulo"] else 0,
            "n_palabras_descripcion": len(anuncio["descripcion"].split()) if anuncio["descripcion"] else 0,
            "tiempo_s": anuncio["tiempo_s"],
            "error": error,
        }
    )
    print(f"[{len(resultados)}/{len(testbench)}] {anuncio['titulo']} ({anuncio['tiempo_s']} s)")

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("resultados_testbench.csv", index=False)

print("\n--- Resumen ---")
print(f"JSON válido: {df_resultados['error'].isna().sum()}/{len(df_resultados)}")
print(f"Tiempo promedio por anuncio: {df_resultados['tiempo_s'].mean():.1f} s")
print(f"Palabras en descripción — min: {df_resultados['n_palabras_descripcion'].min()}, "
      f"media: {df_resultados['n_palabras_descripcion'].mean():.0f}, "
      f"max: {df_resultados['n_palabras_descripcion'].max()}")

[1/23] Casa de 2 recámaras en zona residencial (33.0 s)
[2/23] Casa de 2 recámaras en zona tranquila (43.0 s)
[3/23] Departamento de 234 m² en Condominio con 7 Amenidades (38.5 s)
[4/23] Casa de 2 recámaras en zona residencial (43.6 s)
[5/23] Casa de 3 recámaras en zona tranquila (40.6 s)


KeyboardInterrupt: 

## 6. Experimento de temperatura

La misma propiedad con tres temperaturas, para elegir el punto entre fidelidad (baja) y
redacción con más personalidad (alta). Hipótesis: 0.7 es el balance correcto.

In [16]:
draft_fijo = draft_a_texto(testbench.iloc[2])
print(draft_fijo, "\n" + "=" * 70)

for temp in (0.3, 0.7, 1.0):
    anuncio = generar_anuncio(draft_fijo, temperature=temp)
    print(f"\n### temperature = {temp} ({anuncio['tiempo_s']} s)")
    print(f"TÍTULO: {anuncio['titulo']}")
    print(anuncio["descripcion"])

Tipo de propiedad: Departamento
Superficie: 234 m²
Precio de lista: $6,035,620 MXN
Recámaras: 2
Baños: 2 baños
Estacionamientos: 1
Año de construcción: 2007 (19 años de antigüedad)
En condominio: sí
Número de amenidades: 7 

### temperature = 0.3 (16.9 s)
TÍTULO: Departamento de 234 m² en condominio
Departamento de 234 m² con 2 recámaras y 2 baños, ubicado en un condominio con 7 amenidades. Construido en 2007, cuenta con estacionamiento para un auto. Ideal para quienes buscan un espacio cómodo y bien ubicado en un entorno seguro y con servicios comunes.

### temperature = 0.7 (17.6 s)
TÍTULO: Departamento de 234 m² en condominio
Departamento de 234 m² ubicado en un condominio con 7 amenidades. Cuenta con 2 recámaras y 2 baños, distribuidos de forma funcional y cómoda. Posee 1 estacionamiento y fue construido en 2007, con 19 años de antigüedad. Ideal para quienes buscan un espacio equilibrado y cercano a servicios comunes.

### temperature = 1.0 (15.5 s)
TÍTULO: Departamento de 2 recáma

## 7. Evaluación manual

Revisar `resultados_testbench.csv` y marcar:

- [ ] **Fidelidad:** ¿algún anuncio inventó características que no están en el draft?
      (ubicaciones, amenidades específicas, "cerca de…", vistas, acabados)
- [ ] **Idioma:** ¿el español suena natural y de la región? (recámaras ✓, habitaciones ✗;
      estacionamientos ✓, cajones de estacionamiento ✗, plaza de garaje ✗)
- [ ] **Longitud:** títulos ≤ 10 palabras y descripciones entre 120 y 200 en ≥ 90% de los casos
- [ ] **JSON:** válido en 23/23
- [ ] **Tiempo:** promedio por anuncio aceptable para el flujo asíncrono (≤ 60 s en el VPS)

### Bitácora de versiones del prompt

| Versión | Cambio | Resultado |
|---|---|---|
| v1 | Prompt inicial con regla anti-invención | JSON válido 5/5+; español natural; invención leve de entorno ("zona residencial", "buen acceso a servicios"); usa "cajones de estacionamiento" (copiado del draft) |
| v2 | Vocabulario regional: "estacionamientos" en draft y prompt (regla 7); temperature default 1.0 (preferida en el experimento de la sección 6) | |

**Candidato a v3:** reforzar la regla 1 contra adjetivos de entorno/ubicación no presentes en el
draft ("zona residencial", "entorno seguro", "buen acceso") — aparecieron en las 3 temperaturas.

**Siguientes pasos** (fuera de este notebook): benchmark en el VPS, contenedor `llama-server`,
worker consumidor de `llm_queue`, y enriquecer el esquema del draft con ubicación y amenidades
con nombre.